In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)


def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i+h, j:j+w] * K).sum()
    return Y


X_sample = torch.randn(5,5)
K_edge = torch.tensor([[1,0,-1],[1,0,-1],[1,0,-1]], dtype=torch.float32)
print(corr2d(X_sample, K_edge))
print(F.conv2d(X_sample.unsqueeze(0).unsqueeze(0), K_edge.unsqueeze(0).unsqueeze(0)))


class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(16, 120, 5), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(120, 84), nn.ReLU(),
            nn.Linear(84, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model_cnn = LeNet5().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.001)

def train_epoch(model, loader, opt):
    model.train()
    total_loss, correct = 0, 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        opt.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        opt.step()
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    return total_loss/len(loader), correct/len(loader.dataset)

for epoch in range(10):
    loss, acc = train_epoch(model_cnn, train_loader, optimizer)
    print(f"Epoch {epoch+1}: Loss={loss:.4f}, Acc={acc:.4f}")


model_cnn.eval()
correct=0
with torch.no_grad():
    for data,target in test_loader:
        data,target = data.to(device),target.to(device)
        output = model_cnn(data)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
print(f"Test Accuracy: {correct/len(test_set):.4f}")


def viz_layer1(model, img_tensor):
    model.eval()
    with torch.no_grad():
        x = img_tensor.unsqueeze(0).to(device)
        out = model.features[0](x)
    fig, axs = plt.subplots(1, 6, figsize=(12,4))
    for i in range(6):
        axs[i].imshow(out[0,i].cpu(), cmap='gray')
        axs[i].axis('off')
    plt.show()

sample_img, _ = test_set[0]
viz_layer1(model_cnn, sample_img)


100%|██████████| 26.4M/26.4M [00:02<00:00, 11.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 204kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.76MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 18.7MB/s]


tensor([[-2.8450, -1.3760,  2.7358],
        [-2.7797, -0.7872,  0.5119],
        [-3.5298,  3.2759,  1.1428]])
tensor([[[[-2.8450, -1.3760,  2.7358],
          [-2.7797, -0.7872,  0.5119],
          [-3.5298,  3.2759,  1.1428]]]])


NameError: name 'device' is not defined